# Validate Stage 2 handoff indexes

Coordinator entry point. Read `docs/stage2_interfaces.md` at the published base.
Mount the intended Drive account in Colab before setup. This notebook does not mount,
authenticate to Hugging Face, install GPU packages, load weights, fit heads or run workers.
Install the CPU environment with `pip install -r requirements-stage2-cpu.txt` if needed;
record actual versions. Never update the checkout midway through a run.
Worker notebooks consume the printed plan; per-runtime Gemma authentication remains required
in model-loading notebooks. A passed plan describes readiness and inventory, not arrays.


In [ ]:
from pathlib import Path
import json, os, subprocess, sys

REPO = Path('/content/Research')  # Clean checkout at the published immutable base.
PHASE1_ROOT = Path('/content/drive/MyDrive/phase1')
if not (REPO / 'src/stage2/interfaces.py').is_file():
    raise ValueError('Set REPO to the shared-foundation checkout before continuing')
if not PHASE1_ROOT.is_dir():
    raise ValueError('Mount the intended Drive account and set PHASE1_ROOT')
os.chdir(REPO)
CODE_COMMIT = subprocess.check_output(['git','rev-parse','HEAD'], text=True).strip()
if subprocess.check_output(['git','status','--porcelain'], text=True).strip():
    raise ValueError('Use a clean frozen checkout; do not pull during a job')
print(json.dumps({'repo': str(REPO), 'phase1_root': str(PHASE1_ROOT), 'code_commit': CODE_COMMIT}, indent=2))


In [ ]:
# Every reference is explicit; an empty inventory reports blocked, never complete.
INVENTORY = {
    'data_plan': 'REPLACE_WITH_PRINTED_SHARED_PLAN/_COMPLETE.json',
    'model_plans': {},  # {'qwen': '/.../_COMPLETE.json', 'gemma': '/.../_COMPLETE.json'}
    'feature_releases': {},  # {'qwen': [12 condition/split manifests], 'gemma': [...]}
    'resource_reports': {},  # One measured first-complete-chunk report per model.
    'text_families': [],  # 13 complete family indexes; partial indexes remain blocked.
}
STORAGE_SCOPE = 'persistent-drive'


In [ ]:
import tempfile
from src.stage2.schema import marker_path
if 'REPLACE' in INVENTORY['data_plan']:
    raise ValueError('Set the explicit shared data plan manifest')
marker_path(INVENTORY['data_plan'], PHASE1_ROOT)
print(json.dumps(INVENTORY,indent=2))
with tempfile.TemporaryDirectory() as temporary:
    path=Path(temporary)/'inventory.json'
    path.write_text(json.dumps(INVENTORY),encoding='utf-8')
    completed=subprocess.run([sys.executable,'-m','src.stage2.operations','validate',
        '--phase1-root',str(PHASE1_ROOT),'--inventory',str(path),
        '--storage-scope',STORAGE_SCOPE],text=True,capture_output=True)
    print(completed.stdout)
    if completed.returncode:
        print(completed.stderr)
        raise RuntimeError('Stage 2 validation blocked; inspect the diagnostic')
